# Phase 5 — Deployment-faithful binder retrieval: results

**Question (north star).** MaSIF's surface fingerprints are tuned to *bound* crystal structures and degrade
on the AI-predicted structures you'd actually search at deployment. Phase 5 asks, at scale on the real
domain: **given a target interface — experimental or AI-predicted — can a from-scratch SE(3)-invariant
encoder retrieve the true binding partner better and more robustly than frozen MaSIF?**

**Key insight:** a structure bound to a *different* partner is effectively *apo* w.r.t. an *unknown* binder,
so conformation-robustness *is* the deployment task.

Every figure below is drawn inline from the gate JSONs in `logs/phase5/`.

## Background — the model under test & data provenance  *(read this first)*

**Model evaluated (the "learned" encoder):** `ret_full_ctr_best.pt`
— absolute path `/work/upthomae/Meng/phase4/ret_full_ctr_best.pt` (a Phase-4 artifact; model files are
git-ignored, so it lives on `/work`, not in the repo tree).

It is the **from-scratch heterogeneous GNN** (atom nodes + surface-vertex nodes, 3 invariant edge types),
SE(3)-invariant by construction. Built in two stages, both on the **MaSIF-search training set**:
1. **VICReg contrastive pretrain** (Stage-A) → `vicreg_sc_best_seed0.pt`;
2. **retrieval fine-tune** with a chain-level contrastive objective + **DC-offset centering** (60 epochs,
   dense patch) → `ret_full_ctr_best.pt`.

**Training / validation data**
- **Train:** `retrieval_train_ids.txt` = **4872 complexes** (the preprocessed MaSIF-search training set,
  `stageA_full_npz`). This set is **complex-level, NOT sequence-cluster-deduplicated internally.**
- **Val (best-epoch selection during training):** `m2_eval_ids.txt` = 31 complexes (the Phase-4 AF3 set).
- **Frozen baseline** = MaSIF's own pretrained 80-D descriptor net (`masif-neosurf-af2`), the exact ceiling,
  scored on identical patches. (It was trained by the MaSIF authors on their training set; it is fixed here.)

**Leakage control for Phase 5 (important, and corrected after review).**
The encoder's actual training list (`retrieval_train_ids.txt`) turned out to **include 60 test-list
complexes** (a Phase-4 split imperfection). So the Phase-5 evaluation set is filtered to be **disjoint from
the encoder's *actual* training data** — no exact-id members and no ≥30%-identity sequence-cluster homologs —
**and** deduplicated within itself. Net: **959 nominal test → 287 truly-held-out (269 with AF3-apo).** All
gate numbers below are on this leak-free set (`logs/phase5/eval_sc304_clean_vs_enc.txt`,
`gate_fullclean_*.json`). The learned encoder was **never trained or validated on any complex it is scored on
here, nor on any sequence homolog of one.**

## What was built (the pipeline)

| stage | what | artifact |
|---|---|---|
| **Split** | MaSIF-search test list → **sequence-cluster-clean** (mmseqs2 30% id, disjoint from train) → dedup | `logs/phase5/eval_sc304.txt` |
| **Holo** | experimental structures → reference `.sif` surfaces + 80-D descriptors → hetero-graph npz | `*__holo__*` |
| **Apo** | self-generated **AF3** monomer per side (unbound-like) → surfaces → npz | `*__af3__*` |
| **Gate** | 4-cell query×DB retrieval (holo/af3 × holo/af3), learned vs frozen on identical patches, DC-offset centered | `gate_full_{pos,pos_sc}.json` |
| **Ablation** | no-atom-graph encoder (chem graph removed) through the gate | `gate_noaa_*` |

**Metrics:** top-k recall + median rank of the true partner; **robustness = drop from holo→holo**; controls =
shuffled-partner (chance) + frozen-on-identical-patches (exact ceiling). Patches: `pos_sc` (sc-gated,
MaSIF-favourable) and `pos` (dense interface, deployment-realistic).

### Setup — imports, colorblind-safe palette, and load the gate results

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

ROOT = "/scratch/ymeng/masif-graph"

# Okabe-Ito colorblind-safe palette: identity by entity (never cycled)
LEARNED, FROZEN, CHANCE, NOAA = "#0072B2", "#E69F00", "#9AA0A6", "#56B4E9"
CELLS = ["HH", "AH", "HA", "AA"]
CELL_LABEL = {"HH":"HH\nholo\u2192holo","AH":"AH\naf3\u2192holo","HA":"HA\nholo\u2192af3","AA":"AA\naf3\u2192af3"}

plt.rcParams.update({
    "figure.dpi":120, "font.size":11,
    "axes.spines.top":False, "axes.spines.right":False,
    "axes.grid":True, "grid.color":"#E6E8EB", "grid.linewidth":0.8, "axes.axisbelow":True,
    "axes.edgecolor":"#B0B4B8",
})

def load(patch):
    """Load a gate result JSON (returns None if not produced yet)."""
    for stem in ("gate_fullclean","gate_full"):   # prefer the leak-free set
        f=f"{ROOT}/logs/phase5/{stem}_{patch}.json"
        if os.path.exists(f): return json.load(open(f))
    return None

def barlabels(ax, bars, fmt="{:.2f}", dy=0.005):
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h+dy, fmt.format(h), ha="center", va="bottom",
                fontsize=8.5, color="#333")

gate = {p: load(p) for p in ("pos_sc","pos")}
{p:(None if d is None else f"DB={d['db_chains']} n={d['n']}") for p,d in gate.items()}

In [ ]:
# Summary metrics: learned vs frozen, both patches
import pandas as pd
rows=[]
for patch,d in gate.items():
    if d is None: continue
    r=d["results"]
    for c in CELLS:
        for m in ("frozen","learned"):
            e=r[f"{c}_{m}"]
            rows.append(dict(patch=patch, cell=c, method=m, top5=round(e["top5"],2), med_rank=int(e["median_rank"])))
df=pd.DataFrame(rows)
df.pivot_table(index=["patch","cell"], columns="method", values=["top5","med_rank"])

## 1. The evaluation set — leakage-controlled
The nominal MaSIF-search test set (959) is **62% train homologs**. A 30%-identity sequence-cluster holdout +
within-test dedup leaves **304** truly-held-out complexes (**284** with AF3-apo). This is the honest denominator.

In [ ]:
# Eval-set construction (leak-free): remove seq-cluster homologs of the ENCODER's training set + dedup
steps = ["MaSIF-search\ntest list", "held-out & non-redundant\n(30% seq-id vs encoder train)", "with AF3-apo\n(usable in gate)"]
vals  = [959, 287, 269]
fig, ax = plt.subplots(figsize=(7.6,3.0))
y = np.arange(len(vals))[::-1]
ax.barh(y, vals, color=["#B0B4B8","#4E8FBF",LEARNED], height=0.6)
for yi,v in zip(y,vals): ax.text(v+8, yi, str(v), va="center", fontsize=10, color="#333")
ax.set_yticks(y); ax.set_yticklabels(steps, fontsize=9.5)
ax.set_xlim(0,1050); ax.set_xlabel("complexes"); ax.grid(axis="y", visible=False)
ax.set_title("Phase-5 eval set: 70% of the nominal test set removed as train-homologs / redundant / exact-train leaks",
             loc="left", weight="bold", fontsize=10.5)
ax.annotate("\u2212672 (incl. 16 exact members\nof the encoder's training set)", xy=(287,y[1]),
            xytext=(470,y[1]+0.3), fontsize=8.5, color="#B00",
            arrowprops=dict(arrowstyle="->", color="#B00", lw=1.1))
plt.tight_layout(); plt.show()

## 2. The gate — does the learned encoder retrieve the true binder?
Four query×DB regimes. **HH** (holo→holo) = do-no-harm floor; **AA** (af3→af3) = real deployment (an
AI-predicted database queried by an AI-predicted target). Higher top-5 recall = better.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(11.5,4.3), sharey=True)
for ax,(pk,title) in zip(axes, [("pos_sc","sc-gated patch"),("pos","dense interface patch (deployment-realistic)")]):
    d = gate[pk]; r = d["results"]; x = np.arange(4); w = 0.38
    fz = [r[f"{c}_frozen"]["top5"] for c in CELLS]; lr = [r[f"{c}_learned"]["top5"] for c in CELLS]
    b1 = ax.bar(x-w/2, fz, w, color=FROZEN, label="frozen MaSIF")
    b2 = ax.bar(x+w/2, lr, w, color=LEARNED, label="learned (invariant)")
    barlabels(ax,b1); barlabels(ax,b2)
    ax.axhline(5/d["db_chains"], ls=(0,(4,3)), color=CHANCE, lw=1.2)
    ax.text(3.4, 5/d["db_chains"]+0.01, "chance", color="#666", fontsize=8, ha="right")
    ax.set_xticks(x); ax.set_xticklabels([CELL_LABEL[c] for c in CELLS], fontsize=9)
    ax.set_title(f"{title}\nDB={d['db_chains']}, n={d['n']}", loc="left", fontsize=10.5)
    ax.set_ylim(0,0.8); ax.grid(axis="x", visible=False)
axes[0].set_ylabel("top-5 recall (true partner in top 5)")
axes[0].legend(frameon=False, loc="upper right", fontsize=9.5)
fig.suptitle("The gate: does the learned encoder retrieve the true binder? (higher = better)", x=0.02, ha="left", weight="bold", fontsize=12.5)
plt.tight_layout(rect=(0,0,1,0.94)); plt.show()

**Read it:** learned (blue) beats frozen (orange) in **every** cell and both patches. On the **dense**
patch frozen is barely above chance while learned is strong — and learned's **AA** bar (fully AI-predicted)
is as high as its **HH** bar: *no degradation from the conformation shift.*

## 3. Frozen MaSIF collapses on realistic (dense) interfaces
Median rank of the true partner (log; **1** = perfect). Frozen only works on its favourable sc-gated patch;
on the dense interface it falls to ~random (median rank >100 of 568) while learned holds **rank 1**.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(11.5,4.3), sharey=True)
for ax,(pk,title) in zip(axes, [("pos_sc","sc-gated"),("pos","dense (deployment)")]):
    d = gate[pk]; r = d["results"]; x = np.arange(4); w = 0.38
    fz = [r[f"{c}_frozen"]["median_rank"] for c in CELLS]; lr = [r[f"{c}_learned"]["median_rank"] for c in CELLS]
    b1 = ax.bar(x-w/2, fz, w, color=FROZEN, label="frozen MaSIF")
    b2 = ax.bar(x+w/2, lr, w, color=LEARNED, label="learned")
    barlabels(ax,b1,"{:.0f}",0.02); barlabels(ax,b2,"{:.0f}",0.02)
    ax.set_yscale("log"); ax.set_ylim(0.8, max(fz)*2+5)
    ax.axhline(d["db_chains"]/2, ls=(0,(4,3)), color=CHANCE, lw=1.2)
    ax.text(3.4, d["db_chains"]/2*1.05, "chance", color="#666", fontsize=8, ha="right")
    ax.set_xticks(x); ax.set_xticklabels([CELL_LABEL[c] for c in CELLS], fontsize=9)
    ax.set_title(f"{title} patch (DB={d['db_chains']})", loc="left", fontsize=10.5); ax.grid(axis="x", visible=False)
axes[0].set_ylabel("median rank of true partner (log; 1=best)")
axes[0].legend(frameon=False, loc="upper left", fontsize=9.5)
fig.suptitle("Frozen MaSIF collapses to ~random on dense interfaces; learned stays at rank 1", x=0.02, ha="left", weight="bold", fontsize=12.5)
plt.tight_layout(rect=(0,0,1,0.94)); plt.show()

## 4. Conformation-robustness (the north star)
Robustness = how much top-5 recall **drops** from holo→holo to an AF3-involving cell (smaller = better).
Learned barely moves (≈0 on the fully-predicted AA cell); frozen loses a lot.

In [ ]:
groups=[]
for pk,pl in [("pos_sc","sc-gated"),("pos","dense")]:
    rob = gate[pk]["robustness"]
    for c in ["AH","HA","AA"]:
        groups.append((f"{pl}\n{c}", rob[f"frozen_{c}_drop"]["top5"], rob[f"learned_{c}_drop"]["top5"]))
x = np.arange(len(groups)); w = 0.38
fig, ax = plt.subplots(figsize=(8.2,4.2))
b1 = ax.bar(x-w/2, [g[1] for g in groups], w, color=FROZEN, label="frozen MaSIF")
b2 = ax.bar(x+w/2, [g[2] for g in groups], w, color=LEARNED, label="learned")
barlabels(ax,b1,"{:+.2f}",0.002); barlabels(ax,b2,"{:+.2f}",0.002)
ax.axhline(0, color="#666", lw=1)
ax.set_xticks(x); ax.set_xticklabels([g[0] for g in groups], fontsize=8.5)
ax.set_ylabel("top-5 recall DROP from holo\u2192holo\n(smaller = more robust)")
ax.set_title("Robustness to AI-predicted structures: learned barely degrades, frozen loses a lot", loc="left", weight="bold", fontsize=11.5)
ax.legend(frameon=False, loc="upper left", fontsize=9.5); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

## 5. Retrieval curve — the fully AI-predicted case (AA, dense)
Fraction of queries with the true partner in the top-k, for the AA cell on the dense patch — the exact
deployment scenario. Learned jumps to ~0.6 at k=1; frozen hugs the shuffled-control (chance) line.

In [ ]:
d = gate["pos"]; r = d["results"]; ks = np.arange(1,31)
fig, ax = plt.subplots(figsize=(7.4,4.4))
for key,color,lab,lw,ls in [("AA_learned",LEARNED,"learned  AA (af3\u2192af3)",2.4,"-"),
                            ("AA_frozen",FROZEN,"frozen  AA (af3\u2192af3)",2.4,"-"),
                            ("HH_frozen_shuffled",CHANCE,"shuffled control",1.6,"--")]:
    ranks = np.array(r[key]["ranks"])
    ax.plot(ks, [(ranks<=k).mean() for k in ks], color=color, lw=lw, ls=ls, label=lab)
ax.set_xlabel("k  (rank cutoff)"); ax.set_ylabel("fraction with true partner in top-k")
ax.set_xlim(1,30); ax.set_ylim(0,1)
ax.set_title("Retrieval curve \u2014 fully AI-predicted query & database (dense patch)", loc="left", weight="bold", fontsize=11.5)
ax.legend(frameon=False, loc="lower right", fontsize=9.5)
plt.tight_layout(); plt.show()

## 6. Graph ablation — is the atom graph the source of the advantage?
A no-atom-graph encoder (chem/covalent edges removed, otherwise identical recipe) run through the same gate.
Phase-4 found the atom graph adds no robustness (3× null); this tests it on real retrieval. *(This cell draws
once `logs/phase5/gate_noaa_pos.json` exists — the no-aa encoder trains in the background.)*

In [ ]:
dn = load("pos"); dnoaa = json.load(open(f"{ROOT}/logs/phase5/gate_noaaclean_pos.json")) if os.path.exists(f"{ROOT}/logs/phase5/gate_noaaclean_pos.json") else None
if dnoaa is None:
    print("no-aa ablation gate not ready yet \u2014 re-run this cell later.")
else:
    x = np.arange(4); w = 0.38
    full = [dn["results"][f"{c}_learned"]["top5"] for c in CELLS]
    noaa = [dnoaa["results"][f"{c}_learned"]["top5"] for c in CELLS]
    fig, ax = plt.subplots(figsize=(8.2,4.2))
    b1 = ax.bar(x-w/2, full, w, color=LEARNED, label="learned  +atom graph")
    b2 = ax.bar(x+w/2, noaa, w, color=NOAA, label="learned  no atom graph")
    barlabels(ax,b1); barlabels(ax,b2)
    ax.set_xticks(x); ax.set_xticklabels([CELL_LABEL[c] for c in CELLS], fontsize=9)
    ax.set_ylabel("top-5 recall (dense patch)"); ax.set_ylim(0,0.8); ax.grid(axis="x", visible=False)
    ax.set_title("Graph ablation: does the atom/chem graph earn its keep?", loc="left", weight="bold", fontsize=11.5)
    ax.legend(frameon=False, loc="upper right", fontsize=9.5)
    plt.tight_layout(); plt.show()

## Verdict — GATE MET (decisively)

On a sequence-cluster-clean held-out set with self-generated AF3-apo structures, the from-scratch invariant
encoder **(1)** beats frozen MaSIF on the holo→holo do-no-harm floor, **(2)** is conformation-robust
(holo→AA top-5 drop ≈ 0 vs frozen +0.03–0.15), and **(3)** dominates the fully-AI-predicted AA deployment
cell (dense: median rank **1**, top-5 **0.64** vs frozen 0.06). Frozen MaSIF's usable retrieval needs the
semi-oracular sc patch and collapses on realistic dense interfaces.

**The learned encoder is the better and more robust deployment retriever on AI-predicted structures — the
Phase-5 north star.** Full write-up: `docs/15-phase5-results.md`.